# Stats Scratch — Original DeLong / AUC Approach

**Status:** Preserved for reference. Not the current analysis path.

## Why this is parked

The original `delong_analysis.ipynb` trained the 3 CNNs and then compared them via DeLong's test on ROC-AUC. That's defensible — AUC only needs ranking scores, and regression outputs work fine as ranking scores — but it's an indirect way to ask the question we actually care about.

Our models are **regressors** trained with MSE on pIC50. The most direct comparison is on regression error itself (Wilcoxon signed-rank on per-molecule absolute errors, with Holm correction for the 3 pairwise tests), not on a derived binary classification AUC. We may also try a stacked metalearner to see whether the models contribute complementary signal.

The new flow:
- `train_and_save.ipynb` (Colab + GPU) — trains models, saves `predictions.csv` and `model_metrics.csv`
- A separate local stats notebook — does Wilcoxon, residual analysis, metalearner, etc.

## What's in this file

The DeLong implementation, the pairwise runner with Holm correction, and the ROC curve plot — all kept here in case we want to revisit AUC as a secondary check.

## How to run it

These cells assume you have `predictions.csv` from `train_and_save.ipynb`. The cell below loads it; the rest of the cells were copied verbatim from the original notebook and reference `c_test`, `results`, etc., so you'd need to reconstruct those from the CSV before running.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

# Adjust path as needed
PRED_CSV = 'results/predictions.csv'
preds = pd.read_csv(PRED_CSV)
preds.head()

## DeLong Test — AUC Comparison (preserved from original)

Uses pIC50 predictions as scores for binary classification (Class = active/inactive).
DeLong test asks: is the AUC difference between two models statistically significant?

In [ ]:
# DeLong test implementation (Sun & Xu, 2014)
# No extra dependencies needed — just numpy + scipy
from scipy.stats import norm

def delong_test(y_true, pred_a, pred_b):
    """Two-sided DeLong test comparing AUCs of two models.

    Returns: auc_a, auc_b, z_stat, p_value
    """
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)

    pos = y_true == 1
    neg = y_true == 0
    x_p = np.vstack([pred_a[pos], pred_b[pos]])  # (2, m)
    x_n = np.vstack([pred_a[neg], pred_b[neg]])  # (2, n)
    m = x_p.shape[1]
    n = x_n.shape[1]

    # Structural components (placements) via Mann-Whitney U
    v_10 = np.zeros((2, m))
    v_01 = np.zeros((2, n))
    aucs = np.zeros(2)

    for k in range(2):
        v_10[k] = np.array([
            np.mean((x_n[k] < xi) + 0.5 * (x_n[k] == xi)) for xi in x_p[k]
        ])
        v_01[k] = np.array([
            np.mean((x_p[k] > xj) + 0.5 * (x_p[k] == xj)) for xj in x_n[k]
        ])
        aucs[k] = np.mean(v_10[k])

    s10 = np.cov(v_10)
    s01 = np.cov(v_01)
    s = s10 / m + s01 / n

    diff = aucs[0] - aucs[1]
    var_diff = s[0, 0] + s[1, 1] - 2 * s[0, 1]
    if var_diff <= 0:
        return aucs[0], aucs[1], 0.0, 1.0
    z = diff / np.sqrt(var_diff)
    p = 2 * norm.sf(abs(z))

    return aucs[0], aucs[1], z, p

In [ ]:
# Pairwise DeLong with Holm-Bonferroni correction
# Expects `preds` columns: pIC50_actual, Class, and one pred_<model> column per model.
pred_cols = [c for c in preds.columns if c.startswith('pred_')]
y_class = preds['Class'].values

delong_rows = []
for i in range(len(pred_cols)):
    for j in range(i + 1, len(pred_cols)):
        a, b = pred_cols[i], pred_cols[j]
        auc_a, auc_b, z, p = delong_test(y_class, preds[a].values, preds[b].values)
        delong_rows.append({
            'model_a': a, 'model_b': b,
            'auc_a': round(auc_a, 4), 'auc_b': round(auc_b, 4),
            'z_stat': round(z, 4), 'p_value': p,
        })

k = len(delong_rows)
order = sorted(range(k), key=lambda i: delong_rows[i]['p_value'])
running_max = 0.0
p_holm = [None] * k
for rank, idx in enumerate(order):
    adj = (k - rank) * delong_rows[idx]['p_value']
    running_max = max(running_max, adj)
    p_holm[idx] = min(running_max, 1.0)

for row, p_adj in zip(delong_rows, p_holm):
    row['p_holm'] = round(p_adj, 6)
    row['p_value'] = round(row['p_value'], 6)
    row['sig'] = '***' if p_adj < 0.001 else '**' if p_adj < 0.01 else '*' if p_adj < 0.05 else 'ns'

pd.DataFrame(delong_rows)

## ROC Curves (preserved from original)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for col in pred_cols:
    fpr, tpr, _ = roc_curve(y_class, preds[col])
    auc = roc_auc_score(y_class, preds[col])
    ax.plot(fpr, tpr, label=f'{col} (AUC={auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Architecture Comparison')
ax.legend()
fig.tight_layout()
plt.show()